# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-6-luna'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-6 Luna figure out which links are relevant

### Use a call to GPT-6 Luna to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [5]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [6]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-6 Luna

In [7]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Nanbeige/Nanbeige4.3-33B
Updated
4 days ago
•
13.8k
•
864
deepseek-ai/DeepSeek-V4.1-Flash-1017
Updated
15 days ago
•
156k
•
4.1k
LiquidAI/LFM3-1.7B
Updated
3 days ago
•
32.9k
•
626
nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B
Updated
2 days ago
•
24.4k
•
542
mistralai/Mistral-Large-4-1T-

In [8]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [9]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nNanbeige/Nanbeige4.3-33B\nUpdated\n4 days ago\n•\n13.8k\n•\n865\ndeepseek-ai/DeepSeek-V4.1-Flash-1017\nUpdated\n15 days a

In [10]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    print('Creating brochure for ' + company_name + ' at ' + url)
    result = response.choices[0].message.content
    return result

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future — a collaboration platform where machine learning practitioners, researchers, and developers come together to create, share, and advance open-source models, datasets, and applications.

As the home of machine learning innovation, Hugging Face offers a centralized Hub that hosts over 2 million machine learning models and 500,000+ datasets, empowering data scientists and AI enthusiasts around the globe.

---

## What We Offer

- **Models:** Access and contribute to millions of state-of-the-art AI models spanning NLP, computer vision, speech recognition, and more.
- **Datasets:** Discover, share, and work with a vast collection of datasets to accelerate your research and AI development.
- **Spaces:** Build and showcase machine learning applications directly on the platform, leveraging powerful infrastructure such as RTX PRO 6000 GPUs.
- **Enterprise Solutions:** Tailored AI deployments, including enterprise support, inference endpoints, storage buckets, and collaboration tools to help businesses integrate AI seamlessly.
- **Community & Learning:** Engage with a thriving global community via forums, Discord, GitHub, and ongoing educational resources like blog posts, daily papers, and tutorials to stay on the cutting edge.
- **HuggingChat:** Community-driven chatbots and conversational AI as part of our expansive offerings.

---

## Our Culture

Hugging Face embraces openness, collaboration, and transparency. The company’s mission centers on building an inclusive AI ecosystem where knowledge and innovation are shared freely. As a community-first organization, Hugging Face encourages active participation across all levels — from hobbyists to industry-leading experts.

---

## Customers & Community

- Academic researchers pushing the boundaries of AI.
- Companies across sectors leveraging AI models for NLP, vision, and speech.
- Open-source developers building new applications and AI capabilities.
- Enthusiasts and students accessing resources to learn and grow.
- Enterprise clients benefiting from scalable AI deployment and expert support.

The platform's success is amplified by its engaged community contributing to the continuous improvement and expansion of models, datasets, and AI tools.

---

## Careers at Hugging Face

Hugging Face is an exciting place to grow your career in AI. The company seeks passionate individuals in areas such as:

- Machine Learning Engineering
- Research and Development
- Software Engineering
- Community and Developer Relations
- Enterprise Sales and Support

Join a dynamic, innovative team dedicated to transforming AI collaboration worldwide. Hugging Face values creativity, diversity, and impact, offering opportunities to work with cutting-edge AI technologies and shape the future.

---

## Connect & Explore

- **Website:** [huggingface.co](https://huggingface.co)  
- **Community:** Join discussions on [Discord](https://discord.gg/huggingface) and [Forums](https://discuss.huggingface.co)  
- **GitHub:** Contribute or explore projects at [github.com/huggingface](https://github.com/huggingface)  
- **Learn:** Access tutorials, blog posts, and papers for continuous learning  
- **Enterprise:** Discover solutions for scalable AI integration  

---

Hugging Face — Empowering the global AI community to build the future together.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [11]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

    return response

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 18 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future of machine learning. It is a leading collaboration platform where researchers, developers, and enterprises come together to create, discover, and improve machine learning models, datasets, and applications. With over 2 million models and 500,000+ datasets, Hugging Face hosts one of the world’s largest ML communities dedicated to open innovation.

---

## What We Offer

- **Models:** Access and contribute to over 2 million machine learning models across various domains including NLP, vision, audio, and more.
- **Datasets:** Explore and share diverse datasets with more than 500k datasets available for training and research.
- **Spaces:** Host and run applications powered by machine learning models effortlessly on Hugging Face’s infrastructure.
- **Enterprise Solutions:** Tailored enterprise support including hosted endpoints, inference infrastructure, and enterprise-grade security.
- **Developer Tools:** APIs, SDKs, and tools such as Inference Endpoints, Storage Buckets, and Hugging Face PRO to accelerate AI development.
- **Community & Learning:** Active forums, Discord channels, blogs, and daily research paper discussions to foster continual learning and collaboration.

---

## Our Customers and Collaborators

Hugging Face serves a vibrant community including:

- Researchers in academia and industry pushing the boundaries of AI.
- Enterprises deploying robust AI solutions with scalability and compliance.
- Developers building innovative AI-powered applications.
- AI enthusiasts and data scientists seeking shared resources and collaboration.

Notable organizations and projects actively use or contribute models and datasets, boosting innovation across fields such as natural language processing, computer vision, security, and 3D asset search.

---

## Company Culture

At Hugging Face, collaboration and open innovation are at the heart of everything. The culture fosters inclusivity, continuous learning, and community engagement. The team values transparency, empowers developers worldwide, and promotes sharing knowledge freely. Hugging Face encourages creativity and supports contributors in pushing the AI frontier collectively.

---

## Career Opportunities

Hugging Face is growing rapidly and invites passionate individuals to join their team to help build the future of AI. Opportunities exist for:

- Machine Learning Engineers
- Data Scientists
- Software Developers
- Community Managers
- Enterprise Solution Architects

Working at Hugging Face means being part of a mission-driven company that empowers the world’s largest ML community and tackles challenging problems with cutting-edge technology and open collaboration.

---

## Connect with Hugging Face

- Explore Models & Datasets: [huggingface.co/models](https://huggingface.co/models)  
- Join the Community: Discord, Forum, and GitHub for real-time collaboration  
- Learn & Develop: Extensive documentation and hands-on tutorials  
- Enterprise Inquiries: Tailored solutions for business needs through Hugging Face PRO and enterprise support  

---

## Branding

- Official Colors: #FFD21E, #FF9D00, #6B7280  
- Logos and brand assets are available for use in your own AI projects.

---

**Hugging Face - The Home of Machine Learning**  
Join the AI community that builds the future together.

---

In [12]:
import gradio as gr

In [13]:
def stream_brochure_to_interface(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response


In [ ]:
company_name_input = gr.Textbox(label='Company Name', info='Enter the company name you want to include on the brochure', lines=7)
url_input = gr.Textbox(label='Websit URL', info='Please enter the URL of the company\'s home page', lines=7)
brochure_output = gr.Markdown(label='Generated Brochure:')

webpage = gr.Interface(
    fn=stream_brochure_to_interface,
    title='Automatic Brochure Generator',
    inputs=[company_name_input, url_input],
    outputs=[brochure_output],
    examples=[
        ['Hugging Face', 'https://huggingface.co']
    ],
    flagging_mode='never'
)

webpage.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 9 relevant links
Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 8 relevant links
Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 7 relevant links


In [26]:
def translate_brochure(company_name, url, language):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": 'Please translate the brochure in the user prompt into the following language: ' + language + ', and keep the original markdown format in your response.'},
            {"role": "user", "content": create_brochure(company_name, url)}
        ],
        stream=True
    )
    print('Translating...')
    response = ''
    display_handle = display(Markdown(''), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    

In [27]:
translate_brochure("HuggingFace", "https://huggingface.co", "Simplified Chinese")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links
Creating brochure for HuggingFace at https://huggingface.co
Translating...


# Hugging Face：构建未来的 AI 社区

---

## 关于 Hugging Face

Hugging Face 是一个充满活力且富有创新精神的 AI 社区，处于机器学习的前沿。公司致力于打造 AI 的未来，通过一个开放平台，汇聚成千上万的开发者、研究人员和企业，共同合作开发模型、数据集和 AI 应用。

他们的平台托管了超过 200 万个模型、50 万个数据集和超过 100 万个 AI 驱动的应用程序，成为全球创作者的机器学习之家。

---

## 平台与解决方案

- **模型：** 访问并共享涵盖众多 AI 任务的大量预训练机器学习模型库。
- **数据集：** 协作构建、共享并利用广泛的数据集来训练 AI 模型。
- **Spaces：** 直接在 Hugging Face 平台上托管和部署交互式机器学习应用。
- **Buckets & Storage（存储桶与存储）：** 无缝管理可扩展的 AI 数据和存储基础设施。
- **企业方案：** 为团队和企业定制的计划，包括 Hugging Face PRO，侧重于增强支持、推理端点、定制部署及可扩展基础设施。
- **推理服务提供商：** 高性能、可扩展的推理服务，助力将 AI 模型集成到生产环境。
- **社区与学习：** 通过论坛、Discord 频道、博客文章、每日论文和 GitHub 开源贡献，参与 AI 社区互动。

---

## 客户与社区

Hugging Face 服务涵盖独立开发者、AI 研究人员，以及追求前沿 AI 解决方案的大型企业。该平台通过提供协作工具和基础设施，帮助用户快速创新并轻松部署 AI 应用。

他们的社区遍布全球，包容性强，对开放 AI 充满热情，并通过分享与协作，积极推动机器学习的发展。

---

## 公司文化

- **协作创新：** Hugging Face 基于开放协作生长，使 AI 变得可访问，通过社区贡献不断推进技术。
- **透明与开放：** 公司重视开源原则，致力于让机器学习工具和资源实现民主化。
- **社区驱动：** 注重营造一个欢迎创新思想，在共享知识和实际应用中成长的环境。
- **前沿技术：** 不断挑战极限，进行创新研究和构建可扩展的 AI 基础设施。

---

## Hugging Face 职业发展

Hugging Face 欢迎对 AI、机器学习和开源技术充满热情的优秀人才。加入这里意味着：

- 成为一家快速成长、处于前沿科技领域，深度融入 AI 生态系统的公司一员。
- 与多元化的全球 AI 专家、工程师和研究人员团队协作。
- 职位涵盖工程、研究、产品开发及企业支持等多个领域。
- 公司文化强调创造力、影响力、社区参与及持续学习。

探索 Hugging Face 的职业机会，一起与一个有远见且包容的社区共同塑造 AI 的未来。

---

## 为什么选择 Hugging Face？

- **最大的共享 AI 资源：** 访问全球最大量的公开机器学习模型和数据集集散地。
- **强大且可扩展的基础设施：** 支持从研究原型到企业级部署，提供专属支持。
- **社区与企业支持：** 为个人贡献者和企业客户提供专属资源，最大化 AI 价值。
- **端到端 AI 解决方案：** 覆盖模型开发、部署、管理和应用——一站式服务。

---

## 加入 AI 的未来

无论您是开发者、企业 CIO、投资者，还是 AI 爱好者，Hugging Face 都提供工具、社区和生态，助您在 AI 领域创新并成功。

**了解更多：** https://huggingface.co  
**加入社区：** Discord | GitHub | 论坛

---

*Hugging Face — 构建未来的 AI 社区。*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>